# Task 1: Install required Python libraries

In [ ]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)

# Task 2: Download the dataset

In [ ]:
from pathlib import Path
import requests
import pandas as pd

# Create data directory
Path("data").mkdir(exist_ok=True)

# Download the dataset if it doesn't exist
csv_path = Path("data/results.csv")
if not csv_path.exists():
    url = "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge/hands-on-labs/main/02_football_lab_june/04_data/results.csv"
    response = requests.get(url)
    csv_path.write_bytes(response.content)

# Load the dataset
matches = pd.read_csv(csv_path, parse_dates=["date"])

# Print information
print(matches.shape)
print(matches["date"].min())
print(matches["date"].max())
matches.head(3)

# Task 3: Explore the data

In [ ]:
# Top 10 most frequent tournaments
print("Top 10 Most Frequent Tournaments:")
print(matches["tournament"].value_counts().head(10))
print("\n" + "="*50 + "\n")

# Top 15 teams by total matches played
print("Top 15 Teams by Total Matches Played:")
home_counts = matches["home_team"].value_counts()
away_counts = matches["away_team"].value_counts()
total_counts = home_counts.add(away_counts, fill_value=0).sort_values(ascending=False)
print(total_counts.head(15))
print("\n" + "="*50 + "\n")

# Number of matches per decade
print("Number of Matches per Decade:")
matches["decade"] = (matches["date"].dt.year // 10) * 10
decade_counts = matches.groupby("decade").size().sort_index()
for decade, count in decade_counts.items():
    print(f"{decade}s: {count}")

# Task 4: Engineer features for the model

In [ ]:
import pandas as pd

# Filter and sort matches from 1990 onward
filtered = matches[matches["date"] >= "1990-01-01"].copy()
filtered = filtered.sort_values("date").reset_index(drop=True)

# Helper functions
def winrate(hist):
    if len(hist) == 0:
        return 0.5
    return sum(w for _, _, w in hist) / len(hist)

def goal_avg(hist):
    if len(hist) == 0:
        return 1.0
    return sum(gf for gf, _, _ in hist) / len(hist)

def recent_form(hist):
    if len(hist) < 10:
        return 0.5
    recent = hist[-10:]
    return sum(w for _, _, w in recent) / len(recent)

# Major tournaments
major_tournaments = {
    "Soccer World Cup",
    "Soccer World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations"
}

# Initialize team history dictionary
team_history = {}

# Build features
rows = []
for idx, row in filtered.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    
    # Get history (empty list if team not seen before)
    home_hist = team_history.get(home, [])
    away_hist = team_history.get(away, [])
    
    # Compute features from current history
    team_a_winrate = winrate(home_hist)
    team_b_winrate = winrate(away_hist)
    team_a_goal_avg = goal_avg(home_hist)
    team_b_goal_avg = goal_avg(away_hist)
    team_a_recent_form = recent_form(home_hist)
    team_b_recent_form = recent_form(away_hist)
    is_neutral = int(row["neutral"])
    is_major_tournament = 1 if row["tournament"] in major_tournaments else 0
    
    # Determine outcome
    if home_score > away_score:
        outcome = 0
    elif home_score == away_score:
        outcome = 1
    else:
        outcome = 2
    
    # Append feature row
    rows.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "team_a_winrate": team_a_winrate,
        "team_b_winrate": team_b_winrate,
        "team_a_goal_avg": team_a_goal_avg,
        "team_b_goal_avg": team_b_goal_avg,
        "team_a_recent_form": team_a_recent_form,
        "team_b_recent_form": team_b_recent_form,
        "is_neutral": is_neutral,
        "is_major_tournament": is_major_tournament,
        "outcome": outcome
    })
    
    # Update history AFTER computing features
    home_won = 1 if home_score > away_score else 0
    away_won = 1 if away_score > home_score else 0
    
    if home not in team_history:
        team_history[home] = []
    if away not in team_history:
        team_history[away] = []
    
    team_history[home].append((home_score, away_score, home_won))
    team_history[away].append((away_score, home_score, away_won))

# Create DataFrame
features_df = pd.DataFrame(rows)

# Output
print(features_df.shape)
features_df.head(3)

# Task 5: Split data into training and test sets

In [ ]:
# Define feature columns
feature_cols = [
    "team_a_winrate",
    "team_b_winrate",
    "team_a_goal_avg",
    "team_b_goal_avg",
    "team_a_recent_form",
    "team_b_recent_form",
    "is_neutral",
    "is_major_tournament"
]

# Split by date: training set before 2018-01-01, test set on or after 2018-01-01
train_df = features_df[features_df["date"] < "2018-01-01"]
test_df = features_df[features_df["date"] >= "2018-01-01"]

# Create X_train, X_test, y_train, y_test
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df["outcome"]
y_test = test_df["outcome"]

# Print shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("\nClass distribution in y_train:")
print(y_train.value_counts(normalize=True))

# Task 6: Train and evaluate the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

# Create and train the model
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate and print test accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Calculate and print baseline accuracy (most frequent class)
most_frequent_class = y_train.value_counts().idxmax()
baseline_accuracy = (y_train == most_frequent_class).mean()
print(f"Baseline Accuracy (always predict most frequent class): {baseline_accuracy * 100:.2f}%")

# Print confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ["Home win", "Draw", "Away win"]
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("\nConfusion Matrix (rows=actual, columns=predicted):")
print(cm_df)

# Print feature importances sorted descending
importances = model.feature_importances_
feature_importance_pairs = list(zip(feature_cols, importances))
feature_importance_pairs.sort(key=lambda x: x[1], reverse=True)
print("\nFeature Importances (sorted descending):")
for feature, importance in feature_importance_pairs:
    print(f"{feature}: {importance:.4f}")

# Task 7: Save the model and team statistics

In [ ]:
from pathlib import Path
import joblib
import pandas as pd

# Step 1: Create models directory
Path("models").mkdir(exist_ok=True)

# Build a set of soccer teams (teams that appear in World Cup qualification)
wc_qual_matches = matches[matches["tournament"] == "Soccer World Cup qualification"]
soccer_teams = set(wc_qual_matches["home_team"]) | set(wc_qual_matches["away_team"])

# Build team_stats dictionary
team_stats = {}

for team in matches["home_team"].unique():
    # Skip if not a soccer team
    if team not in soccer_teams:
        continue
    
    # Get all matches for this team
    home_matches = matches[matches["home_team"] == team]
    away_matches = matches[matches["away_team"] == team]
    
    total_matches = len(home_matches) + len(away_matches)
    
    # Skip teams with fewer than 30 matches
    if total_matches < 30:
        continue
    
    # Calculate wins
    home_wins = (home_matches["home_score"] > home_matches["away_score"]).sum()
    away_wins = (away_matches["away_score"] > away_matches["home_score"]).sum()
    total_wins = home_wins + away_wins
    winrate = total_wins / total_matches
    
    # Calculate goals scored
    home_goals = home_matches["home_score"].sum()
    away_goals = away_matches["away_score"].sum()
    total_goals = home_goals + away_goals
    goal_avg = total_goals / total_matches
    
    # Calculate recent form (last 10 matches by date)
    all_team_matches = pd.concat([
        home_matches.assign(is_home=True, goals_for=home_matches["home_score"], goals_against=home_matches["away_score"]),
        away_matches.assign(is_home=False, goals_for=away_matches["away_score"], goals_against=away_matches["home_score"])
    ]).sort_values("date")
    
    if len(all_team_matches) < 10:
        recent_form = 0.5
    else:
        last_10 = all_team_matches.tail(10)
        recent_wins = (last_10["goals_for"] > last_10["goals_against"]).sum()
        recent_form = recent_wins / 10
    
    # Store team stats
    team_stats[team] = {
        "winrate": float(winrate),
        "goal_avg": float(goal_avg),
        "recent_form": float(recent_form),
        "matches_played": int(total_matches)
    }

# Step 2: Save the model and team data
joblib.dump(model, "models/match_predictor.pkl")
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols}, "models/team_data.pkl")

# Print summary
print(f"Number of teams stored: {len(team_stats)}")

# Top 5 teams by winrate (with >= 100 matches)
teams_100plus = {team: stats for team, stats in team_stats.items() if stats["matches_played"] >= 100}
top_5 = sorted(teams_100plus.items(), key=lambda x: x[1]["winrate"], reverse=True)[:5]

print("\nTop 5 teams by winrate (with >= 100 matches):")
for i, (team, stats) in enumerate(top_5, 1):
    print(f"{i}. {team}: {stats['winrate']:.3f} ({stats['matches_played']} matches)")

# Task 8: Try the model

In [ ]:
import pandas as pd

def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    """
    Predict the outcome of a match between team_a and team_b.
    
    Args:
        team_a: Name of team A (home team)
        team_b: Name of team B (away team)
        is_neutral: Whether the match is on neutral ground (default: True)
        is_major_tournament: Whether it's a major tournament (default: True)
    
    Returns:
        Dictionary with win probabilities for team_a, draw, and team_b
    """
    # Check if teams are in team_stats
    if team_a not in team_stats:
        raise ValueError(f"Team '{team_a}' not found in team statistics. Please check the team name.")
    if team_b not in team_stats:
        raise ValueError(f"Team '{team_b}' not found in team statistics. Please check the team name.")
    
    # Get team statistics
    stats_a = team_stats[team_a]
    stats_b = team_stats[team_b]
    
    # Build feature row
    feature_row = pd.DataFrame([{
        "team_a_winrate": stats_a["winrate"],
        "team_b_winrate": stats_b["winrate"],
        "team_a_goal_avg": stats_a["goal_avg"],
        "team_b_goal_avg": stats_b["goal_avg"],
        "team_a_recent_form": stats_a["recent_form"],
        "team_b_recent_form": stats_b["recent_form"],
        "is_neutral": int(is_neutral),
        "is_major_tournament": int(is_major_tournament)
    }])
    
    # Reindex to match feature_cols order
    feature_row = feature_row[feature_cols]
    
    # Get prediction probabilities
    proba = model.predict_proba(feature_row)
    
    # Map probabilities: 0 = team_a win, 1 = draw, 2 = team_b win
    return {
        "team_a_win_prob": float(proba[0][0]),
        "draw_prob": float(proba[0][1]),
        "team_b_win_prob": float(proba[0][2])
    }

# Test the function
print("Prediction for Brazil vs Argentina:")
result1 = predict_match("Brazil", "Argentina")
print(f"  Brazil win: {result1['team_a_win_prob']:.2%}")
print(f"  Draw: {result1['draw_prob']:.2%}")
print(f"  Argentina win: {result1['team_b_win_prob']:.2%}")

print("\nPrediction for Germany vs Brazil:")
result2 = predict_match("Germany", "Brazil")
print(f"  Germany win: {result2['team_a_win_prob']:.2%}")
print(f"  Draw: {result2['draw_prob']:.2%}")
print(f"  Brazil win: {result2['team_b_win_prob']:.2%}")

# Task 9: Create an interactive UI application

In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

# Page configuration
st.set_page_config(
    page_title="Soccer 2026 Match Predictor",
    page_icon="⚽",
    layout="centered"
)

@st.cache_resource
def load_artifacts():
    """Load the trained model and team data."""
    model = joblib.load("models/match_predictor.pkl")
    team_data = joblib.load("models/team_data.pkl")
    team_stats = team_data["team_stats"]
    feature_cols = team_data["feature_cols"]
    return model, team_stats, feature_cols

# Load artifacts
model, team_stats, feature_cols = load_artifacts()

# Title and caption
st.title("⚽ Soccer 2026 Match Predictor")
st.caption("Predictions based on historical international football results")

# Sort team names
team_names = sorted(team_stats.keys())

# Team selection
col1, col2 = st.columns(2)
with col1:
    default_a = team_names.index("Brazil") if "Brazil" in team_names else 0
    team_a = st.selectbox("Team A", team_names, index=default_a)
with col2:
    default_b = team_names.index("Argentina") if "Argentina" in team_names else 1
    team_b = st.selectbox("Team B", team_names, index=default_b)

# Match settings
is_neutral = st.checkbox("Neutral venue", value=True)
is_major_tournament = st.checkbox("Major tournament (e.g. World Cup)", value=True)

# Predict button
if st.button("Predict", type="primary", use_container_width=True):
    if team_a == team_b:
        st.error("Please pick two different teams.")
    else:
        # Get team statistics
        stats_a = team_stats[team_a]
        stats_b = team_stats[team_b]
        
        # Build feature row
        feature_row = pd.DataFrame([{
            "team_a_winrate": stats_a["winrate"],
            "team_b_winrate": stats_b["winrate"],
            "team_a_goal_avg": stats_a["goal_avg"],
            "team_b_goal_avg": stats_b["goal_avg"],
            "team_a_recent_form": stats_a["recent_form"],
            "team_b_recent_form": stats_b["recent_form"],
            "is_neutral": int(is_neutral),
            "is_major_tournament": int(is_major_tournament)
        }])
        
        # Reindex to match feature_cols order
        feature_row = feature_row[feature_cols]
        
        # Get prediction probabilities
        proba = model.predict_proba(feature_row)
        team_a_prob = proba[0][0]
        draw_prob = proba[0][1]
        team_b_prob = proba[0][2]
        
        # Display metrics
        st.subheader("Prediction Results")
        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric(f"{team_a} wins", f"{team_a_prob * 100:.1f}%")
        with col2:
            st.metric("Draw", f"{draw_prob * 100:.1f}%")
        with col3:
            st.metric(f"{team_b} wins", f"{team_b_prob * 100:.1f}%")
        
        # Display progress bars
        st.progress(team_a_prob, text=f"{team_a} win probability")
        st.progress(draw_prob, text="Draw probability")
        st.progress(team_b_prob, text=f"{team_b} win probability")
        
        # Display team statistics table
        st.subheader("Team Statistics")
        stats_table = pd.DataFrame({
            "Team": [team_a, team_b],
            "Win Rate": [f"{stats_a['winrate']:.3f}", f"{stats_b['winrate']:.3f}"],
            "Avg Goals Scored": [f"{stats_a['goal_avg']:.2f}", f"{stats_b['goal_avg']:.2f}"],
            "Recent Form (Last 10)": [f"{stats_a['recent_form']:.3f}", f"{stats_b['recent_form']:.3f}"],
            "Matches Played": [stats_a["matches_played"], stats_b["matches_played"]]
        })
        st.table(stats_table)

Writing app.py


# Task 10: Launch the UI application

In [13]:
import subprocess
import sys
import time
import webbrowser

# Start Streamlit as a background process
streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.headless",
        "true",
        "--server.port",
        "8501",
        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for server to start
print("Starting Streamlit server...")
time.sleep(4)

# Open in browser
webbrowser.open("http://localhost:8501")

# Print confirmation
print("Streamlit app is now running at: http://localhost:8501")
print("To stop the server later, run: streamlit_process.terminate()")

Starting Streamlit server...
Streamlit app is now running at: http://localhost:8501
To stop the server later, run: streamlit_process.terminate()
